In [1]:
import warnings

In [2]:
warnings.simplefilter(action='ignore')

In [3]:
import pickle
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from lightfm import LightFM
from lightfm.data import Dataset

from replay.metrics import HitRate, NDCG, Coverage, Experiment
from replay.preprocessing.filters import MinCountFilter, LowRatingFilter
from replay.splitters import RandomSplitter

In [4]:
random.seed(42)
np.random.seed(42)

In [5]:
USER_COL = "user_name"
ITEM_COL = "item_title"
RATING_COL = "rating"

In [6]:
with open("artificial_profiles_scores.pkl", "rb") as f:
    interactions = pickle.load(f)

with open("artificial_profiles.json", "r", encoding="utf-8") as f:
    features_by_user = json.load(f)

with open("titles_with_tags_dict.pkl", "rb") as f:
    features_by_item = pickle.load(f)

print("Data loaded successfully:")
print(f"  - Users: {len(interactions)}")
print(f"  - Items: {len(features_by_item)}")
print(f"  - Total interactions: {sum(len(ratings) for ratings in interactions.values())}")

Data loaded successfully:
  - Users: 31
  - Items: 1164
  - Total interactions: 359


In [7]:
sample_user = list(features_by_user.keys())[0]
print(f"Sample user: {sample_user}")
print(f"Bio: {features_by_user[sample_user]['bio'][:200]}...")
print(f"Tags: {features_by_user[sample_user]['tags']}")

Sample user: global_economics_and_geopolitics_analyst
Bio: A highly analytical researcher deeply engaged in monitoring global economic trends, geopolitical risks, international trade, and the socio-economic development of various regions (especially BRICS, As...
Tags: ['global_economy', 'geopolitics', 'macroeconomics', 'international_economics', 'data_analysis', 'forex', 'regional_development', 'brics', 'political_economy', 'international_relations', 'data_monitoring', 'time_series', 'system_thinking', 'geopolitics_of_BRICS', 'investment_analysis', 'economic_forecasting', 'trade_policy', 'market_forecasting', 'data_mining', 'quantitative_finance', 'global_systems', 'regional_analysis', 'economic_development']


In [8]:
sample_item = list(features_by_item.keys())[0]
print(f"Sample item: {sample_item[:80]}...")
print(f"Tags: {features_by_item[sample_item]}")

Sample item: Исследование приоритетов и механизмов реализации отраслевых (секторальных) полит...
Tags: ['international_relations', 'political_economics', 'policy_analysis', 'BRICS', 'geopolitics', 'international_policy', 'comparative_politics']


In [9]:
import numpy as np
import pandas as pd

In [10]:
data = []

for user in interactions:
    for payload in interactions[user]:
        data.append((user, json.loads(payload).get("title"), interactions[user][payload]))

df = pd.DataFrame(data=data, columns=["user_name", "item_title", "rating"])

In [11]:
df.head()

,user_name,item_title,rating
0,global_economics_and_geopolitics_analyst,Группа поддержки профессионального самоопредел...,1
1,global_economics_and_geopolitics_analyst,Правовые аспекты обучения моделей искусственно...,3
2,global_economics_and_geopolitics_analyst,Разработка веб-сайта на конструкторе веб-сайто...,1
3,global_economics_and_geopolitics_analyst,Ценностные паттерны в консервативном сегменте ...,2
4,global_economics_and_geopolitics_analyst,Формирование и развитие маркетинговой стратеги...,2


In [12]:
data = []

for user in features_by_user:
    data.append((user, features_by_user[user]["tags"]))

df_users = pd.DataFrame(data=data, columns=["user_name", "features"])

In [13]:
df_users.head()

,user_name,features
0,global_economics_and_geopolitics_analyst,"[global_economy, geopolitics, macroeconomics, ..."
1,international_relations_and_geopolitics_expert,"[international_politics, geopolitics, regional..."
2,multilingual_linguistics_researcher,"[linguistics, translation_technology, corpus_p..."
3,education_and_cultural_development_expert,"[language_teaching, educational_materials, cul..."
4,sociocultural_anthropologist_and_politician,"[social_anthropology, political_analysis, cult..."


In [14]:
data = []

for item in features_by_item:
    data.append((item, features_by_item[item]))

df_items = pd.DataFrame(data=data, columns=["item_title", "features"])

In [15]:
df_items.head()

,item_title,features
0,Исследование приоритетов и механизмов реализац...,"[international_relations, political_economics,..."
1,Антрополе - научно-популярный видео-подкаст о ...,"[anthropology, social_phenomena, media, podcas..."
2,"Разработка, создание и ведение сайта, посвящен...","[web_development, history, cultural_studies, d..."
3,Перевод с английского языка коллективной моног...,"[criminology, literature_review, translation, ..."
4,Сеть военно-политических союзов в Евразии: баз...,"[geopolitics, international_relations, databas..."


In [16]:
dfg = df.groupby(by=ITEM_COL).agg({USER_COL: "nunique"}).reset_index()
dfg.columns = ["item_title", "user_name_count"]
dfg.sort_values(by="user_name_count", ascending=False)

,item_title,user_name_count
271,Театральный киноклуб,3
315,“Пишу тебе”: цифровой корпус почтовых открыток...,2
63,Волонтёрство на XIII ежегодной международной к...,2
246,Создание CRM-системы для торгового дома на меж...,2
247,Создание арт-галереи современного искусства в ...,2
...,...,...
111,Коррупционные риски в госсекторе в России и за...,1
110,Корпоративный венчурный капитал в европейской ...,1
109,Концепт оценки в русскоязычном политическом ди...,1
108,"Конференция ""Исследовательская деятельность уч...",1


In [17]:
df[USER_COL].nunique()

31

In [18]:
df[ITEM_COL].nunique()

316

In [19]:
train, test = RandomSplitter(
    drop_cold_items=True,
    drop_cold_users=True,
    query_column=USER_COL,
    item_column=ITEM_COL,
    test_size=0.2,
).split(df)

In [20]:
ALL_USERS = train[USER_COL].unique().tolist()
ALL_ITEMS = train[ITEM_COL].unique().tolist()

users = df_users[df_users[USER_COL].isin(ALL_USERS)]
items = df_items[df_items[ITEM_COL].isin(ALL_ITEMS)]

In [21]:
dataset = Dataset()

In [22]:
dataset.fit_partial(ALL_USERS, ALL_ITEMS)

In [23]:
user_mapping = dataset.mapping()[0]
item_mapping = dataset.mapping()[2]

inv_user_mapping = {v:k for k,v in user_mapping.items()}
inv_item_mapping = {v:k for k,v in item_mapping.items()}

In [24]:
train_interactions, train_weights = dataset.build_interactions(train[[USER_COL, ITEM_COL, RATING_COL]].values)

In [25]:
train_interactions

<COOrdinate sparse matrix of dtype 'int32'
	with 287 stored elements and shape (31, 261)>

In [26]:
train_weights

<COOrdinate sparse matrix of dtype 'float32'
	with 287 stored elements and shape (31, 261)>

In [27]:
class LightFM4Rec:
    def __init__(self, model, user_mapping, item_mapping, inv_user_mapping, inv_item_mapping, user_col, item_col, rating_col):
        self.model = model
        self.user_mapping = user_mapping
        self.item_mapping = item_mapping
        self.inv_user_mapping = inv_user_mapping
        self.inv_item_mapping = inv_item_mapping
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col
    
    def fit(self, rating_matrix, user_features=None, item_features=None, epochs=16):
        self.user_features = user_features
        self.item_features = item_features
        self.model.fit(
            rating_matrix,
            user_features=self.user_features,
            item_features=self.item_features,
            epochs=epochs,
        )
    
    def _get_lfm_pred(self, user_id):
        pred = self.model.predict(
            user_ids=user_id,
            item_ids=self.item_ids,
            user_features=self.user_features,
            item_features=self.item_features,
        )
        return pred

    def predict(self, test, interaction_matrix, filter_seen=True, k=10):
        user_ids = test[self.user_col].map(self.user_mapping).unique()
        self.item_ids = list(self.item_mapping.values())
    
        pred = pd.DataFrame(user_ids, columns=[USER_COL])
        scores = np.vstack(pred[USER_COL].apply(self._get_lfm_pred).values)

        if filter_seen:
            scores += np.nan_to_num(interaction_matrix.todense()[user_ids] * -np.inf)
        
        ind_part = np.argpartition(scores, -k)[:, -k:].copy()
        scores_not_sorted = np.take_along_axis(scores, ind_part, axis=1)
        ind_sorted = np.argsort(scores_not_sorted, axis=1)
        scores_sorted = np.sort(scores_not_sorted, axis=1)
        indices = np.take_along_axis(ind_part, ind_sorted, axis=1)

        preds = pd.DataFrame(
            {
                self.user_col: user_ids,
                self.item_col: np.flip(indices, axis=1).tolist(),
                self.rating_col: np.flip(scores_sorted, axis=1).tolist(),
            }
        ).explode([self.item_col, self.rating_col])
        preds[self.user_col] = preds[self.user_col].map(self.inv_user_mapping)
        preds[self.item_col] = preds[self.item_col].map(self.inv_item_mapping)
        
        return preds

In [28]:
model = LightFM4Rec(
    LightFM(
        random_state=42,
        loss="warp",
        no_components=16,
    ),
    user_mapping,
    item_mapping,
    inv_user_mapping,
    inv_item_mapping,
    USER_COL,
    ITEM_COL,
    RATING_COL
)

In [29]:
model.fit(train_weights)

In [30]:
preds_wo_features = model.predict(test, train_interactions)

In [31]:
preds_wo_features.head()

,user_name,item_title,rating
0,education_and_cultural_development_expert,«ISU Buddy» – программа адаптационного сопрово...,-0.076855
0,education_and_cultural_development_expert,Просветительский проект «Голос Востока»,-0.106299
0,education_and_cultural_development_expert,Разработка системы распределения затрат при со...,-0.108147
0,education_and_cultural_development_expert,Школьное математическое образование первой чет...,-0.141734
0,education_and_cultural_development_expert,Влияние Индо-Тихоокеанской стратегии США на Вь...,-0.156206


In [32]:
preds_wo_features[preds_wo_features["user_name"] == 'financial_economist_and_analyst']

,user_name,item_title,rating
6,financial_economist_and_analyst,Научно-учебный центр публичных коммуникаций и ...,-0.015683
6,financial_economist_and_analyst,Базовый курс Малайско-индонезийского языка,-0.081905
6,financial_economist_and_analyst,Создание и редактирование контента для студенч...,-0.102003
6,financial_economist_and_analyst,Создание арт-галереи современного искусства в ...,-0.138129
6,financial_economist_and_analyst,Российская Федерация и страны Субсахарской Афр...,-0.188746
6,financial_economist_and_analyst,Разработка продвинутой интерактивной платформы...,-0.197905
6,financial_economist_and_analyst,Международный мониторинг ответственного бизнеса,-0.212989
6,financial_economist_and_analyst,Применение современных переводческих технологи...,-0.214409
6,financial_economist_and_analyst,Брендинг рынка Чорсу в Ташкенте,-0.275955
6,financial_economist_and_analyst,Дискуссионный клуб по публичному праву,-0.308537


In [33]:
user_tags = users["features"].explode().unique()

In [34]:
item_tags = items["features"].explode().unique()

In [35]:
dataset.fit_partial(user_features=user_tags, item_features=item_tags)

In [36]:
sparse_u_features = dataset.build_user_features(
    [[row.user_name, row.features] for row in users.reset_index().itertuples()]
)

In [37]:
sparse_i_features = dataset.build_item_features(
    [[row.item_title, row.features] for row in items.reset_index().itertuples()]
)

In [38]:
model = LightFM4Rec(
    LightFM(
        random_state=42,
        loss="warp",
        no_components=16,
    ),
    user_mapping,
    item_mapping,
    inv_user_mapping,
    inv_item_mapping,
    USER_COL,
    ITEM_COL,
    RATING_COL
)

In [39]:
model.fit(train_weights, sparse_u_features, sparse_i_features)

In [40]:
preds_w_features = model.predict(test, train_interactions)

In [41]:
preds_w_features[preds_w_features["user_name"] == 'financial_economist_and_analyst']

,user_name,item_title,rating
6,financial_economist_and_analyst,Тестовая ВКР по физике,-0.997874
6,financial_economist_and_analyst,Театральный киноклуб,-1.017882
6,financial_economist_and_analyst,Интерактивная карта национальной кухни Пермско...,-1.045667
6,financial_economist_and_analyst,Составление процессуальных документов (Зайцев ...,-1.050013
6,financial_economist_and_analyst,Просветительский проект «Голос Востока»,-1.050792
6,financial_economist_and_analyst,Международный мониторинг ответственного бизнеса,-1.070748
6,financial_economist_and_analyst,Разработка и отладка учебно-методических матер...,-1.077858
6,financial_economist_and_analyst,Японский театр: погружение в культуру через игру,-1.093519
6,financial_economist_and_analyst,Преимущества низковолатильных ETF с точки зрен...,-1.113339
6,financial_economist_and_analyst,Создание арт-галереи современного искусства в ...,-1.114266


In [42]:
K = [10]
metrics = Experiment(
    [
        NDCG(K),
        HitRate(K),
        Coverage(K),
    ],
    test,
    train,
    query_column=USER_COL,
    item_column=ITEM_COL,
    rating_column=RATING_COL
)

In [43]:
metrics.add_result("LFM_wo_features", preds_wo_features)
metrics.add_result("LFM_w_features", preds_w_features)

In [44]:
metrics.results

,NDCG@10,HitRate@10,Coverage@10
LFM_wo_features,0.083333,0.083333,0.340996
LFM_w_features,0.000000,0.000000,0.042146
